# 🕷️ Agent 2 — Scraper Runner
## Executes Playwright scripts, handles failures, stores raw JSON

**What this agent does:**
- Reads `scrape_manifest.json` from Agent 1
- Runs each Playwright script as a subprocess
- Falls back to HTTP scraping if Playwright fails
- Validates and saves raw JSON per competitor

**SDK: Anthropic (Claude) — NOT OpenAI**

**Input ← Agent 1 | Output → Agent 3**

## 1. Install Dependencies

In [ ]:
%pip install anthropic python-dotenv playwright --quiet

## 2. Setup

In [ ]:
import os, json, subprocess, sys, re
import urllib.request
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import anthropic

load_dotenv()
print('ANTHROPIC_API_KEY loaded:', bool(os.getenv('ANTHROPIC_API_KEY')))

BASE_DIR  = Path('.')
DATA_RAW  = BASE_DIR / 'data' / 'raw'
DATA_LOGS = BASE_DIR / 'data' / 'logs'
for d in [DATA_RAW, DATA_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

client = anthropic.Anthropic()
print('Anthropic client ready')

## 3. Check Manifest from Agent 1

In [ ]:
manifest_path = BASE_DIR / 'scrape_manifest.json'
if not manifest_path.exists():
    print('ERROR: scrape_manifest.json not found. Run Agent 1 first.')
else:
    MANIFEST = json.loads(manifest_path.read_text())
    print(f'Manifest loaded: {len(MANIFEST.get("scripts", []))} scripts')
    for s in MANIFEST.get('scripts', []):
        exists = Path(s['script_path']).exists()
        print(f'  {"OK" if exists else "MISSING"}: {s["competitor"]} [{s.get("priority","?")}]')

## 4. Define Tools

In [ ]:
# ── Tool functions ──────────────────────────────────────────────

def get_manifest() -> str:
    """Returns the scrape manifest with all scripts to run."""
    p = BASE_DIR / 'scrape_manifest.json'
    return p.read_text() if p.exists() else json.dumps({'error': 'No manifest found. Run Agent 1 first.'})


def run_scraper_script(competitor_name: str, script_path: str) -> str:
    """Runs a Playwright scraper script as a subprocess."""
    script = Path(script_path)
    if not script.exists():
        return json.dumps({'status': 'failed', 'error': 'Script not found', 'competitor': competitor_name})
    try:
        print(f'  Scraping {competitor_name}...')
        proc = subprocess.run(
            [sys.executable, str(script)],
            capture_output=True, text=True, timeout=120
        )
        out = DATA_RAW / f'{competitor_name.lower().replace(" ", "_")}_raw.json'
        if proc.returncode == 0 and out.exists():
            data = json.loads(out.read_text())
            sections = [s for s in ['pricing','features','homepage_headline'] if data.get(s)]
            return json.dumps({'status': 'success', 'competitor': competitor_name,
                               'output': str(out), 'sections_captured': sections})
        return json.dumps({'status': 'failed', 'competitor': competitor_name,
                           'returncode': proc.returncode, 'stderr': proc.stderr[-400:]})
    except subprocess.TimeoutExpired:
        return json.dumps({'status': 'timeout', 'competitor': competitor_name})
    except Exception as e:
        return json.dumps({'status': 'error', 'competitor': competitor_name, 'error': str(e)})


def run_fallback_scraper(competitor_name: str, url: str) -> str:
    """HTTP fallback scraper when Playwright fails or times out."""
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'})
        with urllib.request.urlopen(req, timeout=15) as r:
            html = r.read().decode('utf-8', errors='ignore')
        h1s = [re.sub(r'<[^>]+>', '', h).strip()
               for h in re.findall(r'<h1[^>]*>(.*?)</h1>', html, re.I | re.S)][:3]
        title_m = re.search(r'<title>(.*?)</title>', html, re.I)
        data = {
            'competitor_name': competitor_name,
            'website': url,
            'scraped_at': datetime.now().isoformat(),
            'scrape_status': 'partial_fallback',
            'homepage_headline': h1s[0] if h1s else '',
            'homepage_usp': h1s,
            'page_title': title_m.group(1).strip() if title_m else '',
            'pricing': {}, 'features': [], 'reviews_summary': {},
            'blog_topics': [], 'job_postings': {},
            'errors': ['Playwright failed or timed out — used HTTP fallback']
        }
        out = DATA_RAW / f'{competitor_name.lower().replace(" ", "_")}_raw.json'
        out.write_text(json.dumps(data, indent=2))
        print(f'  Fallback OK: {competitor_name}')
        return json.dumps({'status': 'partial_fallback', 'output': str(out), 'headline': data['homepage_headline']})
    except Exception as e:
        return json.dumps({'status': 'failed', 'competitor': competitor_name, 'error': str(e)})


def check_scrape_output(competitor_name: str) -> str:
    """Validates output JSON file quality for a competitor."""
    f = DATA_RAW / f'{competitor_name.lower().replace(" ", "_")}_raw.json'
    if not f.exists():
        return json.dumps({'exists': False, 'competitor': competitor_name})
    d = json.loads(f.read_text())
    return json.dumps({
        'exists': True,
        'competitor': competitor_name,
        'status': d.get('scrape_status', 'unknown'),
        'has_pricing': bool(d.get('pricing')),
        'has_features': bool(d.get('features')),
        'has_homepage': bool(d.get('homepage_headline')),
        'error_count': len(d.get('errors', []))
    })


def save_scrape_log(log_data: str) -> str:
    """Saves run summary log for Agent 3 to reference."""
    p = DATA_LOGS / 'scrape_latest.json'
    p.write_text(log_data)
    print('  Scrape log saved')
    return str(p)


# ── Anthropic tool definitions ──────────────────────────────────
TOOLS = [
    {
        "name": "get_manifest",
        "description": "Returns the scrape manifest JSON with all competitor scripts to execute.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "run_scraper_script",
        "description": "Runs a Playwright scraper script as a subprocess for one competitor.",
        "input_schema": {
            "type": "object",
            "properties": {
                "competitor_name": {"type": "string"},
                "script_path": {"type": "string"}
            },
            "required": ["competitor_name", "script_path"]
        }
    },
    {
        "name": "run_fallback_scraper",
        "description": "HTTP fallback scraper when Playwright fails or times out.",
        "input_schema": {
            "type": "object",
            "properties": {
                "competitor_name": {"type": "string"},
                "url": {"type": "string"}
            },
            "required": ["competitor_name", "url"]
        }
    },
    {
        "name": "check_scrape_output",
        "description": "Validates the output JSON file for a competitor — checks existence and data quality.",
        "input_schema": {
            "type": "object",
            "properties": {
                "competitor_name": {"type": "string"}
            },
            "required": ["competitor_name"]
        }
    },
    {
        "name": "save_scrape_log",
        "description": "Saves the run summary log JSON to disk.",
        "input_schema": {
            "type": "object",
            "properties": {
                "log_data": {"type": "string", "description": "JSON string with per-competitor run results"}
            },
            "required": ["log_data"]
        }
    },
]

# ── Tool dispatcher ─────────────────────────────────────────────
TOOL_FNS = {
    "get_manifest":          lambda **k: get_manifest(),
    "run_scraper_script":    lambda **k: run_scraper_script(**k),
    "run_fallback_scraper":  lambda **k: run_fallback_scraper(**k),
    "check_scrape_output":   lambda **k: check_scrape_output(**k),
    "save_scrape_log":       lambda **k: save_scrape_log(**k),
}

print('Agent 2 tools ready:', [t['name'] for t in TOOLS])

## 5. Agentic Loop (Claude)

In [ ]:
def run_claude_agent(system: str, tools: list, tool_fns: dict, prompt: str,
                     model: str = 'claude-haiku-4-5-20251001', max_tokens: int = 4096) -> str:
    """
    Runs a Claude agentic tool-use loop.
    Uses Haiku for Agent 2 — fast and cost-efficient for scraper execution.
    """
    messages = [{"role": "user", "content": prompt}]
    iteration = 0

    while True:
        iteration += 1
        print(f'  [loop {iteration}] calling Claude ({model})...')

        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system,
            tools=tools,
            messages=messages
        )

        if response.stop_reason == 'end_turn':
            return next((b.text for b in response.content if hasattr(b, 'text')), '')

        if response.stop_reason == 'tool_use':
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    print(f'    → tool: {block.name}({list((block.input or {}).keys())})')
                    fn = tool_fns.get(block.name)
                    try:
                        result = fn(**(block.input or {})) if fn else f'Unknown tool: {block.name}'
                    except Exception as e:
                        result = f'Tool error: {e}'
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            return f'Unexpected stop_reason: {response.stop_reason}'

## 6. Run Agent 2

In [ ]:
SYSTEM = """
You are Agent 2 — the Scraper Runner in a competitive intelligence pipeline.

WORKFLOW (follow in order):
1. Call get_manifest() to get all scripts and their paths
2. For each competitor (high priority first, then medium, then low):
   a. Call run_scraper_script(competitor_name, script_path)
   b. If status is failed or timeout: call run_fallback_scraper(competitor_name, url)
   c. Call check_scrape_output(competitor_name) to validate the output file
3. Build a run_summary JSON with status per competitor
4. Call save_scrape_log(run_summary)

RULES:
- Never skip a competitor — always attempt fallback before giving up
- Do not stop the pipeline because one scraper fails
- Report final status (success/partial_fallback/failed) for every competitor
"""

PROMPT = (
    'Execute all scrapers from the manifest. '
    'Run high priority competitors first. '
    'Use HTTP fallback for any failures. '
    'Validate all outputs and save the run log.'
)

print('Running Agent 2 — Scraper Runner (claude-haiku-4-5-20251001)')
print('=' * 60)
t0 = datetime.now()

output = run_claude_agent(
    system=SYSTEM,
    tools=TOOLS,
    tool_fns=TOOL_FNS,
    prompt=PROMPT,
    model='claude-haiku-4-5-20251001',
    max_tokens=4096
)

elapsed = (datetime.now() - t0).seconds
print(f'\nDone in {elapsed}s')
print(output[:600] if output else '(no text output)')

## 7. Verify Raw Data Files

In [ ]:
print('Raw data files:')
files = sorted(DATA_RAW.glob('*_raw.json'))
if not files:
    print('  No files found. Check agent output above.')
for f in files:
    d = json.loads(f.read_text())
    status = d.get('scrape_status', 'unknown')
    sections = [s for s in ['pricing', 'features', 'homepage_headline', 'reviews_summary', 'blog_topics', 'job_postings']
                if d.get(s)]
    print(f'  {f.name}: [{status}] {len(sections)}/6 sections populated')

## ✅ Done — Next: open `03_agent3_analyst.ipynb`